# 04 · Ekstraksi Embedding BLaIR dan RoBERTa-base
**Bab III Subbab 3.6, Bab II Subbab 2.5 (persamaan 2.3–2.4).** Judul dan isi ulasan ditokenisasi sebagai pasangan teks,
isi dipotong dari belakang jika melebihi 512 token, lalu embedding diambil dari token [CLS] lapisan terakhir dan dinormalisasi L2.
RoBERTa-base (pembanding B5) memakai prosedur yang persis sama.

Embedding dihitung **sekali** tanpa gradien lalu dipakai ulang oleh semua eksperimen. Perlu GPU. Jika sesi Colab terputus,
jalankan ulang sel yang sama dan ekstraksi akan dilanjutkan.

**Keluaran:** `emb_index.parquet`, `emb_blair.npy`, `emb_roberta.npy` (float16)

> **Google Colab.** Jalankan sekali per sesi sebelum sel setup:
> ```
> !git clone -b main https://github.com/Vito28/dl-paper.git
> %cd dl-paper/notebooks
> !pip install -q -r ../requirements.txt
> ```
> Sel setup otomatis me-mount Google Drive (izinkan akses saat diminta). **Semua data dan hasil, termasuk
> unduhan mentah, disimpan di `MyDrive/dl-paper`**, bukan di instance. Jika Drive gagal di-mount atau
> terputus, notebook berhenti dengan error. Jalankan sel terakhir (`finish()`) agar semua tulisan tersinkron.

In [ ]:
# Setup: path proyek dan konfigurasi bersama (ubah konfigurasi di notebooks/settings.py)
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
from settings import cfg, SMOKE, finish
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)

In [ ]:
import torch
from hr import embed
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada (akan sangat lambat)")
df = pd.read_parquet(cfg.path("reviews_split.parquet"))
tg = pd.read_parquet(cfg.path("targets.parquet"))
rows = df[df["rid"].isin(tg["rid"])].sort_values("rid").reset_index(drop=True)
rows[["rid"]].to_parquet(cfg.path("emb_index.parquet"), index=False)
print(len(rows), "ulasan akan diubah menjadi embedding")

In [ ]:
# Pastikan ruang Drive cukup SEBELUM ekstraksi (unggahan yang melebihi kuota Drive bisa gagal tanpa error)
import shutil
reps = ["blair"] + (["roberta"] if "B5" in cfg.experiments_to_run() else [])
need = sum(len(rows) * 768 * 2 for r in reps if not cfg.path(f"emb_{r}.npy").exists())
usage = shutil.disk_usage(cfg.out_dir)
print(f"butuh ~{need / 1e9:.1f} GB untuk {reps}, kosong {usage.free / 1e9:.1f} GB")
if usage.total > 0:
    assert usage.free > 1.2 * need, "Ruang penyimpanan tidak cukup untuk berkas embedding"

## 1. BLaIR-RoBERTa-Base

In [ ]:
emb_b = embed.extract_embeddings(rows, cfg.encoders["blair"], cfg.path("emb_blair.npy"), cfg.max_len, cfg.embed_batch)

## 2. RoBERTa-base (B5)

In [ ]:
if "B5" in cfg.experiments_to_run():
    emb_r = embed.extract_embeddings(rows, cfg.encoders["roberta"], cfg.path("emb_roberta.npy"), cfg.max_len, cfg.embed_batch)

## 3. Pemeriksaan
Norma setiap vektor harus ≈ 1. Contoh kemiripan kosinus antara ulasan dalam satu produk dan antarproduk memberi gambaran awal apakah embedding bermakna.

In [ ]:
E = np.load(cfg.path("emb_blair.npy"), mmap_mode="r")
print("bentuk:", E.shape, "| norma (5 baris pertama):", np.linalg.norm(E[:5].astype(np.float32), axis=1).round(3))
rng = np.random.default_rng(0)
pid = rows["parent_asin"].to_numpy()
i = rng.integers(0, len(rows), 2000); j = rng.integers(0, len(rows), 2000)
cos = (E[i].astype(np.float32) * E[j].astype(np.float32)).sum(1)
same = pid[i] == pid[j]
print("kosinus acak antarproduk:", cos[~same].mean().round(3))
grp = rows.groupby("parent_asin").indices
k = [v[:2] for v in grp.values() if len(v) >= 2][:2000]
a, b = np.array(k).T
print("kosinus dua ulasan produk sama:", (E[a].astype(np.float32) * E[b].astype(np.float32)).sum(1).mean().round(3))
for f in ("emb_blair.json", "emb_roberta.json"):
    if cfg.path(f).exists(): print(f, json.loads(cfg.path(f).read_text()))

## Simpan ke Google Drive
Memastikan semua hasil notebook ini sudah tersinkron ke Drive sebelum runtime ditutup.

In [ ]:
finish()